In [1]:
import pandas as pd
import numpy as np
from pathlib import Path


# ============================================================
# 1. FILE PATHS
# ============================================================

BASE_DIR = Path().resolve().parent

DATA_DIR = (
    BASE_DIR
    / "data"
    / "2011_2012"
    / "Demographics"
)

xpt_file = DATA_DIR / "DEMO_G.xpt"
csv_file = DATA_DIR / "DEMO_G_corrected.csv"


# ============================================================
# 2. READ ORIGINAL XPT
# ============================================================

df = pd.read_sas(
    xpt_file,
    format="xport",
    encoding="latin1"
)

#
# Preserve untouched pandas-decoded source.
#
source_df = df.copy()


print("=" * 75)
print("NHANES 2011-2012 DEMO_G XPT -> CSV")
print("=" * 75)

print("Rows:", f"{len(df):,}")
print("Columns:", len(df.columns))


# ============================================================
# 3. EXPECTED STRUCTURE
# ============================================================

EXPECTED_COLUMNS = [
    "SEQN",
    "SDDSRVYR",
    "RIDSTATR",
    "RIAGENDR",
    "RIDAGEYR",
    "RIDAGEMN",
    "RIDRETH1",
    "RIDRETH3",
    "RIDEXMON",
    "RIDEXAGY",
    "RIDEXAGM",
    "DMQMILIZ",
    "DMQADFC",
    "DMDBORN4",
    "DMDCITZN",
    "DMDYRSUS",
    "DMDEDUC3",
    "DMDEDUC2",
    "DMDMARTL",
    "RIDEXPRG",
    "SIALANG",
    "SIAPROXY",
    "SIAINTRP",
    "FIALANG",
    "FIAPROXY",
    "FIAINTRP",
    "MIALANG",
    "MIAPROXY",
    "MIAINTRP",
    "AIALANGA",
    "WTINT2YR",
    "WTMEC2YR",
    "SDMVPSU",
    "SDMVSTRA",
    "INDHHIN2",
    "INDFMIN2",
    "INDFMPIR",
    "DMDHHSIZ",
    "DMDFMSIZ",
    "DMDHHSZA",
    "DMDHHSZB",
    "DMDHHSZE",
    "DMDHRGND",
    "DMDHRAGE",
    "DMDHRBR4",
    "DMDHREDU",
    "DMDHRMAR",
    "DMDHSEDU"
]


if df.columns.tolist() != EXPECTED_COLUMNS:
    raise ValueError(
        "DEMO_G column names/order differ from expected.\n"
        f"Actual: {df.columns.tolist()}"
    )


if len(df) != 9756:
    raise ValueError(
        f"Unexpected row count: {len(df):,}"
    )


if len(df.columns) != 48:
    raise ValueError(
        f"Unexpected column count: {len(df.columns)}"
    )


print(
    "PASS: Structure = 9,756 rows x 48 columns."
)


# ============================================================
# 4. DEFINE DECIMAL VS WHOLE-NUMBER VARIABLES
# ============================================================

#
# IMPORTANT:
#
# These THREE variables contain genuine decimal values
# and MUST NOT be converted to Int64:
#
#   WTINT2YR
#   WTMEC2YR
#   INDFMPIR
#
# No rounding.
# No float_format.
#

DECIMAL_COLUMNS = [
    "WTINT2YR",
    "WTMEC2YR",
    "INDFMPIR"
]


INTEGER_COLUMNS = [
    col
    for col in df.columns
    if col not in DECIMAL_COLUMNS
]


print("\nExpected genuine decimal variables:")

print(
    DECIMAL_COLUMNS
)


# ============================================================
# 5. CHECK KNOWN XPORT TINY-ZERO ARTIFACT
# ============================================================

#
# Known pandas / SAS XPORT zero-decoding artifact:
#
#     5.397605346934028e-79
#
# NEVER perform a blanket replacement without validating
# the exact variable/count pattern.
#

TINY_VALUE = np.float64(
    5.397605346934028e-79
)


numeric_cols = (
    df.select_dtypes(
        include=[np.number]
    )
    .columns
    .tolist()
)


tiny_counts = (
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
)


tiny_total = int(
    tiny_counts.sum()
)


print("\n--- ORIGINAL XPT TINY-ZERO CHECK ---")


print(
    "Tiny artifact occurrences:",
    f"{tiny_total:,}"
)


print("\nTiny values by variable:")


print(
    tiny_counts[
        tiny_counts > 0
    ]
    .sort_values(
        ascending=False
    )
    .to_string()
)


# ============================================================
# 6. VALIDATE EXACT TINY-ZERO PATTERN
# ============================================================

#
# Exact pattern from attached DEMO_G.xpt:
#
# DMDHHSZE    7060
# DMDHHSZA    6184
# DMDHHSZB    4819
# WTMEC2YR     418
# RIDAGEYR     392
# DMDEDUC3     282
# INDFMPIR      80
# RIDAGEMN      34
# RIDEXAGM       8
#
# TOTAL       19277
#
#
# CDC confirms zero is legitimate:
#
# RIDAGEYR:
#     valid range begins at 0
#
# RIDAGEMN:
#     valid range 0-24
#
# RIDEXAGM:
#     valid range 0-239
#
# DMDEDUC3:
#     0 = Never attended / kindergarten only
#
# WTMEC2YR:
#     valid range begins at 0
#
# INDFMPIR:
#     valid range begins at 0
#
# DMDHHSZA:
#     valid range begins at 0
#
# DMDHHSZB:
#     valid range begins at 0
#
# DMDHHSZE:
#     valid range begins at 0
#

EXPECTED_TINY_COUNTS = {
    "RIDAGEYR": 392,
    "RIDAGEMN": 34,
    "RIDEXAGM": 8,
    "DMDEDUC3": 282,
    "WTMEC2YR": 418,
    "INDFMPIR": 80,
    "DMDHHSZA": 6184,
    "DMDHHSZB": 4819,
    "DMDHHSZE": 7060
}


actual_tiny_counts = (
    tiny_counts[
        tiny_counts > 0
    ]
    .astype(int)
    .to_dict()
)


if actual_tiny_counts != EXPECTED_TINY_COUNTS:

    raise ValueError(
        "STOP: Tiny-value pattern differs from "
        "validated DEMO_G pattern.\n"
        f"Expected: {EXPECTED_TINY_COUNTS}\n"
        f"Actual:   {actual_tiny_counts}"
    )


if tiny_total != 19277:

    raise ValueError(
        f"Unexpected total tiny-value count: {tiny_total:,}"
    )


print(
    "\nPASS: Exact 19,277-value "
    "tiny-zero pattern validated."
)


# ============================================================
# 7. RESTORE ONLY VALIDATED ZERO VALUES
# ============================================================

#
# Restore TINY_VALUE to 0 ONLY in the nine
# validated variables.
#

print("\n--- ZERO RESTORATION ---")


correction_total = 0


for col, expected_count in EXPECTED_TINY_COUNTS.items():

    tiny_mask = (
        df[col]
        .eq(TINY_VALUE)
    )


    actual_count = int(
        tiny_mask.sum()
    )


    if actual_count != expected_count:

        raise ValueError(
            f"{col}: unexpected tiny-value count."
        )


    df.loc[
        tiny_mask,
        col
    ] = 0


    correction_total += actual_count


    print(
        f"{col}: "
        f"{actual_count:,} restored to 0"
    )


if correction_total != 19277:

    raise ValueError(
        "Incorrect total zero-restoration count."
    )


print(
    "\nPASS:",
    f"{correction_total:,}",
    "validated zero values restored."
)


# ============================================================
# 8. CREATE EXPECTED CORRECTED SOURCE COPY
# ============================================================

#
# expected_df represents exactly what we expect after
# correcting pandas' XPORT zero-decoding artifacts.
#

expected_df = source_df.copy()


for col in EXPECTED_TINY_COUNTS:

    expected_df.loc[
        expected_df[col].eq(TINY_VALUE),
        col
    ] = 0


# ============================================================
# 9. CONFIRM NO TINY VALUES REMAIN
# ============================================================

remaining_tiny = int(
    df[numeric_cols]
    .eq(TINY_VALUE)
    .sum()
    .sum()
)


if remaining_tiny != 0:

    raise ValueError(
        f"Tiny values remain after correction: "
        f"{remaining_tiny:,}"
    )


print(
    "PASS: No tiny artifacts remain after restoration."
)


# ============================================================
# 10. DETECT GENUINE DECIMAL VARIABLES
# ============================================================

#
# CRITICAL:
#
# Perform this test AFTER restoring zero artifacts
# and BEFORE converting anything to Int64.
#
# Expected:
#
#   WTINT2YR
#   WTMEC2YR
#   INDFMPIR
#

print("\n--- FRACTIONAL VALUE CHECK ---")


fractional_columns = []


for col in numeric_cols:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) == 0:
        continue


    fractional_mask = ~np.isclose(
        values,
        np.round(values),
        rtol=0,
        atol=1e-12
    )


    if fractional_mask.any():

        fractional_columns.append(
            col
        )

        print(
            f"{col}: "
            f"{int(fractional_mask.sum()):,} "
            "fractional observations"
        )


if set(
    fractional_columns
) != set(
    DECIMAL_COLUMNS
):

    raise ValueError(
        "Unexpected genuine decimal-variable pattern.\n"
        f"Expected: {DECIMAL_COLUMNS}\n"
        f"Actual:   {fractional_columns}"
    )


print(
    "PASS: Exact genuine decimal variables confirmed."
)


# ============================================================
# 11. VALIDATE DECIMAL VARIABLES
# ============================================================

#
# DO NOT ROUND THESE.
#
# CDC:
#
# WTINT2YR:
#     3320.8855 - 220233.3152
#     n = 9756
#
# WTMEC2YR:
#     0 - 222579.78343
#     n = 9756
#
# INDFMPIR:
#     0-4.99 = 7656
#     5.00+  = 1260
#     missing = 840
#

print("\n--- DECIMAL VARIABLE CHECKS ---")


# ---------- WTINT2YR ----------

s = df["WTINT2YR"]


print(
    "WTINT2YR range:",
    s.min(),
    "to",
    s.max()
)


if int(
    s.notna()
    .sum()
) != 9756:

    raise ValueError(
        "WTINT2YR non-missing count differs from CDC."
    )


if not s.between(
    3320.8855,
    220233.315202
).all():

    raise ValueError(
        "WTINT2YR contains unexpected values."
    )


print(
    "PASS: WTINT2YR validated."
)


# ---------- WTMEC2YR ----------

s = df["WTMEC2YR"]


print(
    "WTMEC2YR range:",
    s.min(),
    "to",
    s.max()
)


print(
    "WTMEC2YR zeros:",
    int(
        s.eq(0)
        .sum()
    )
)


if int(
    s.notna()
    .sum()
) != 9756:

    raise ValueError(
        "WTMEC2YR non-missing count differs from CDC."
    )


if not s.between(
    0,
    222579.783434
).all():

    raise ValueError(
        "WTMEC2YR contains unexpected values."
    )


if int(
    s.eq(0)
    .sum()
) != 418:

    raise ValueError(
        "WTMEC2YR should contain exactly "
        "418 validated zero weights."
    )


print(
    "PASS: WTMEC2YR validated."
)


# ---------- INDFMPIR ----------

s = df["INDFMPIR"]


print(
    "INDFMPIR range:",
    s.min(),
    "to",
    s.max()
)


print(
    "INDFMPIR zeros:",
    int(
        s.eq(0)
        .sum()
    )
)


if not (
    int(
        s.between(
            0,
            4.99
        ).sum()
    )
    == 7656
    and
    int(
        s.eq(5)
        .sum()
    )
    == 1260
    and
    int(
        s.isna()
        .sum()
    )
    == 840
    and
    int(
        s.eq(0)
        .sum()
    )
    == 80
):

    raise ValueError(
        "INDFMPIR differs from CDC / "
        "validated zero restoration."
    )


print(
    "PASS: INDFMPIR validated."
)


# ============================================================
# 12. CLEAN AND VALIDATE SEQN
# ============================================================

seqn = (
    df["SEQN"]
    .dropna()
)


if not np.all(
    np.isclose(
        seqn,
        np.round(seqn),
        rtol=0,
        atol=1e-12
    )
):

    raise ValueError(
        "SEQN contains unexpected fractional values."
    )


df["SEQN"] = (
    df["SEQN"]
    .astype("Int64")
)


print("\n--- SEQN ---")


print(
    "Minimum:",
    df["SEQN"].min()
)


print(
    "Maximum:",
    df["SEQN"].max()
)


print(
    "Missing:",
    int(
        df["SEQN"]
        .isna()
        .sum()
    )
)


print(
    "Duplicates:",
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    )
)


if not (
    df["SEQN"].min() == 62161
    and
    df["SEQN"].max() == 71916
    and
    int(
        df["SEQN"]
        .isna()
        .sum()
    ) == 0
    and
    int(
        df["SEQN"]
        .duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "SEQN validation failed."
    )


print(
    "PASS: SEQN validated."
)


# ============================================================
# 13. FORMAT ONLY PROVEN WHOLE-NUMBER FIELDS
# ============================================================

#
# Convert all variables EXCEPT:
#
#     WTINT2YR
#     WTMEC2YR
#     INDFMPIR
#
# to nullable Int64.
#
# NO .round() is used.
#

for col in INTEGER_COLUMNS:

    values = (
        df[col]
        .dropna()
        .to_numpy(
            dtype=float
        )
    )


    if len(values) > 0:

        whole_mask = np.isclose(
            values,
            np.round(values),
            rtol=0,
            atol=1e-12
        )


        if not whole_mask.all():

            raise ValueError(
                f"{col} contains genuine decimal values "
                "and cannot safely be converted to Int64."
            )


    df[col] = (
        df[col]
        .astype("Int64")
    )


print(
    "\nPASS: Whole-number demographics fields "
    "converted to Int64."
)

print(
    "PASS: WTINT2YR, WTMEC2YR and INDFMPIR "
    "remain decimal."
)


# ============================================================
# 14. CDC CHECK - DATA RELEASE CYCLE
# ============================================================

if (
    df["SDDSRVYR"]
    .value_counts()
    .to_dict()
    != {
        7: 9756
    }
):

    raise ValueError(
        "SDDSRVYR differs from CDC."
    )


print(
    "PASS: SDDSRVYR = 7 for all records."
)


# ============================================================
# 15. CDC CHECK - INTERVIEW / EXAM STATUS
# ============================================================

EXPECTED_RIDSTATR = {
    1: 418,
    2: 9338
}


if (
    df["RIDSTATR"]
    .value_counts()
    .sort_index()
    .to_dict()
    != EXPECTED_RIDSTATR
):

    raise ValueError(
        "RIDSTATR differs from CDC."
    )


print(
    "PASS: RIDSTATR matches CDC."
)


#
# Important logical check:
#
# Interview-only participants should have zero MEC weight.
#

interview_only = (
    df["RIDSTATR"]
    .eq(1)
)


zero_mec_weight = (
    df["WTMEC2YR"]
    .eq(0)
)


if not np.array_equal(
    interview_only.to_numpy(),
    zero_mec_weight.to_numpy()
):

    raise ValueError(
        "WTMEC2YR zero positions do not match "
        "RIDSTATR interview-only participants."
    )


print(
    "PASS: All 418 WTMEC2YR zeros correspond "
    "exactly to interview-only participants."
)


# ============================================================
# 16. CDC CHECK - GENDER
# ============================================================

if (
    df["RIAGENDR"]
    .value_counts()
    .sort_index()
    .to_dict()
    != {
        1: 4856,
        2: 4900
    }
):

    raise ValueError(
        "RIAGENDR differs from CDC."
    )


print(
    "PASS: RIAGENDR matches CDC."
)


# ============================================================
# 17. CDC CHECK - RIDAGEYR
# ============================================================

#
# CDC:
#
# 0-79 = 9393
# 80+  = 363
#
# Attached file:
#
# zero age = 392
#

s = df["RIDAGEYR"]


if not (
    int(
        s.between(
            0,
            79
        ).sum()
    )
    == 9393
    and
    int(
        s.eq(80)
        .sum()
    )
    == 363
    and
    int(
        s.eq(0)
        .sum()
    )
    == 392
    and
    int(
        s.isna()
        .sum()
    )
    == 0
):

    raise ValueError(
        "RIDAGEYR differs from CDC / zero restoration."
    )


print(
    "PASS: RIDAGEYR matches CDC."
)


# ============================================================
# 18. CDC CHECK - RIDAGEMN
# ============================================================

#
# CDC:
#
# 0-24 = 626
# missing = 9130
#
# Attached file:
#
# zero = 34
#

s = df["RIDAGEMN"]


if not (
    int(
        s.between(
            0,
            24
        ).sum()
    )
    == 626
    and
    int(
        s.eq(0)
        .sum()
    )
    == 34
    and
    int(
        s.isna()
        .sum()
    )
    == 9130
):

    raise ValueError(
        "RIDAGEMN differs from CDC / zero restoration."
    )


print(
    "PASS: RIDAGEMN matches CDC."
)


# ============================================================
# 19. CDC CHECK - RACE / ETHNICITY
# ============================================================

EXPECTED_RIDRETH1 = {
    1: 1355,
    2: 1076,
    3: 2973,
    4: 2683,
    5: 1669
}


if (
    df["RIDRETH1"]
    .value_counts()
    .sort_index()
    .to_dict()
    != EXPECTED_RIDRETH1
):

    raise ValueError(
        "RIDRETH1 differs from CDC."
    )


EXPECTED_RIDRETH3 = {
    1: 1355,
    2: 1076,
    3: 2973,
    4: 2683,
    6: 1282,
    7: 387
}


if (
    df["RIDRETH3"]
    .value_counts()
    .sort_index()
    .to_dict()
    != EXPECTED_RIDRETH3
):

    raise ValueError(
        "RIDRETH3 differs from CDC."
    )


print(
    "PASS: RIDRETH1/RIDRETH3 match CDC."
)


# ============================================================
# 20. CDC CHECK - EXAM AGE VARIABLES
# ============================================================

#
# RIDEXAGY:
#
# 2-20 = 3418
# missing = 6338
#

s = df["RIDEXAGY"]


if not (
    int(
        s.between(
            2,
            20
        ).sum()
    )
    == 3418
    and
    int(
        s.isna()
        .sum()
    )
    == 6338
):

    raise ValueError(
        "RIDEXAGY differs from CDC."
    )


print(
    "PASS: RIDEXAGY matches CDC."
)


#
# RIDEXAGM:
#
# 0-239 = 4009
# missing = 5747
#
# Attached XPT:
#
# zero = 8
#

s = df["RIDEXAGM"]


if not (
    int(
        s.between(
            0,
            239
        ).sum()
    )
    == 4009
    and
    int(
        s.eq(0)
        .sum()
    )
    == 8
    and
    int(
        s.isna()
        .sum()
    )
    == 5747
):

    raise ValueError(
        "RIDEXAGM differs from CDC / zero restoration."
    )


print(
    "PASS: RIDEXAGM matches CDC."
)


# ============================================================
# 21. CDC CHECK - DMDEDUC3
# ============================================================

#
# CDC specifically reports:
#
# 0 = Never attended / kindergarten only = 282
#

EXPECTED_DMDEDUC3 = {
    0: 282,
    1: 231,
    2: 213,
    3: 229,
    4: 210,
    5: 204,
    6: 182,
    7: 157,
    8: 167,
    9: 151,
    10: 167,
    11: 150,
    12: 38,
    13: 89,
    14: 8,
    15: 106,
    55: 2,
    66: 13
}


actual = (
    df["DMDEDUC3"]
    .value_counts()
    .sort_index()
    .to_dict()
)


if actual != EXPECTED_DMDEDUC3:

    raise ValueError(
        "DMDEDUC3 differs from CDC."
    )


if int(
    df["DMDEDUC3"]
    .isna()
    .sum()
) != 7157:

    raise ValueError(
        "DMDEDUC3 missing count differs from CDC."
    )


print(
    "PASS: DMDEDUC3 matches CDC."
)


# ============================================================
# 22. CDC CHECK - HOUSEHOLD COMPOSITION ZEROS
# ============================================================

#
# These variables contain the majority of the
# XPORT tiny-zero artifacts.
#
# CDC release ranges:
#
# DMDHHSZA = 0-3
# DMDHHSZB = 0-4
# DMDHHSZE = 0-3
#

EXPECTED_HHSZA = {
    0: 6184,
    1: 2236,
    2: 1064,
    3: 272
}


EXPECTED_HHSZB = {
    0: 4819,
    1: 2170,
    2: 1702,
    3: 743,
    4: 322
}


EXPECTED_HHSZE = {
    0: 7060,
    1: 1594,
    2: 1046,
    3: 56
}


if (
    df["DMDHHSZA"]
    .value_counts()
    .sort_index()
    .to_dict()
    != EXPECTED_HHSZA
):

    raise ValueError(
        "DMDHHSZA differs after zero restoration."
    )


if (
    df["DMDHHSZB"]
    .value_counts()
    .sort_index()
    .to_dict()
    != EXPECTED_HHSZB
):

    raise ValueError(
        "DMDHHSZB differs after zero restoration."
    )


if (
    df["DMDHHSZE"]
    .value_counts()
    .sort_index()
    .to_dict()
    != EXPECTED_HHSZE
):

    raise ValueError(
        "DMDHHSZE differs after zero restoration."
    )


print(
    "PASS: Household-composition zero patterns validated."
)


# ============================================================
# 23. CDC CHECK - HOUSEHOLD / FAMILY SIZE
# ============================================================

if not (
    int(
        df["DMDHHSIZ"]
        .between(
            1,
            6
        )
        .sum()
    )
    == 8936
    and
    int(
        df["DMDHHSIZ"]
        .eq(7)
        .sum()
    )
    == 820
):

    raise ValueError(
        "DMDHHSIZ differs from CDC."
    )


print(
    "PASS: DMDHHSIZ matches CDC."
)


# ============================================================
# 24. VALUE FIDELITY BEFORE EXPORT
# ============================================================

#
# Expected changes from pandas-decoded raw XPT:
#
# ONLY the 19,277 validated tiny values become 0.
#
# Everything else must remain numerically exact.
#

print("\n--- VALUE FIDELITY BEFORE EXPORT ---")


failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    current_values = (
        pd.to_numeric(
            df[col],
            errors="coerce"
        )
        .to_numpy(
            dtype=float,
            na_value=np.nan
        )
    )


    same = np.allclose(
        expected_values,
        current_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        failures.append(
            col
        )


if failures:

    raise ValueError(
        "Unexpected value changes before export:\n"
        + str(failures)
    )


print(
    "PASS: All 48 variables match expected "
    "corrected XPT values exactly."
)


# ============================================================
# 25. EXPORT CSV
# ============================================================

#
# CRITICAL:
#
# NO float_format.
# NO rounding.
#
# The genuine decimals in:
#
#   WTINT2YR
#   WTMEC2YR
#   INDFMPIR
#
# are allowed to serialize naturally.
#

df.to_csv(
    csv_file,
    index=False,
    na_rep=""
)


print("\nCSV created:")

print(
    csv_file
)


# ============================================================
# 26. RE-READ CSV AS RAW TEXT
# ============================================================

#
# Reading as text is useful for checking BOTH:
#
#     numeric fidelity
#     exact CSV formatting
#

raw_csv_df = pd.read_csv(
    csv_file,
    dtype=str,
    keep_default_na=False
)


if raw_csv_df.shape != df.shape:

    raise ValueError(
        "CSV dimensions changed after export."
    )


if (
    raw_csv_df.columns.tolist()
    != EXPECTED_COLUMNS
):

    raise ValueError(
        "CSV column names/order changed."
    )


print(
    "PASS: Exported CSV structure preserved."
)


# ============================================================
# 27. EXACT CORRECTED XPT -> CSV NUMERIC FIDELITY
# ============================================================

print("\n--- XPT -> CSV NUMERIC FIDELITY ---")


csv_failures = []


for col in expected_df.columns:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    raw_values = (
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        )
    )


    exported_values = (
        pd.to_numeric(
            raw_values,
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    same = np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    )


    if not same:

        csv_failures.append(
            col
        )


if csv_failures:

    raise ValueError(
        "Corrected XPT -> CSV differences found in:\n"
        + str(csv_failures)
    )


print(
    "PASS: All 48 CSV columns preserve "
    "the corrected XPT numeric values exactly."
)


# ============================================================
# 28. MISSING VALUE FIDELITY
# ============================================================

missing_mismatches = 0


for col in expected_df.columns:

    expected_missing = (
        expected_df[col]
        .isna()
        .to_numpy()
    )


    exported_missing = (
        raw_csv_df[col]
        .eq("")
        .to_numpy()
    )


    missing_mismatches += int(
        np.sum(
            expected_missing
            !=
            exported_missing
        )
    )


print("\n--- MISSING VALUE CHECK ---")


print(
    "Missing-position mismatches:",
    missing_mismatches
)


if missing_mismatches != 0:

    raise ValueError(
        "Missing-value positions changed."
    )


print(
    "PASS: Missing values preserved exactly."
)


# ============================================================
# 29. FINAL TINY-VALUE CHECK
# ============================================================

tiny_csv_count = 0


for col in EXPECTED_COLUMNS:

    values = pd.to_numeric(
        raw_csv_df[col]
        .where(
            raw_csv_df[col] != "",
            np.nan
        ),
        errors="coerce"
    )


    tiny_csv_count += int(
        values
        .eq(TINY_VALUE)
        .sum()
    )


print("\n--- FINAL TINY VALUE CHECK ---")


print(
    "Tiny values remaining:",
    tiny_csv_count
)


if tiny_csv_count != 0:

    raise ValueError(
        "Tiny XPORT artifacts remain in final CSV."
    )


print(
    "PASS: No tiny-value artifacts remain."
)


# ============================================================
# 30. .0 FORMAT CHECK - INTEGER VARIABLES ONLY
# ============================================================

#
# VERY IMPORTANT:
#
# Do NOT search this entire file for ".0".
#
# Genuine decimal variables exist:
#
#     WTINT2YR
#     WTMEC2YR
#     INDFMPIR
#
# Values like:
#
#     INDFMPIR = 5.0
#
# are legitimate decimal representations.
#
# Only INTEGER_COLUMNS are checked.
#

print("\n--- INTEGER COLUMN .0 CHECK ---")


dot_zero_errors = {}


for col in INTEGER_COLUMNS:

    tokens = raw_csv_df[col]


    bad_mask = (
        tokens.ne("")
        &
        tokens.str.endswith(".0")
    )


    count = int(
        bad_mask.sum()
    )


    if count > 0:

        dot_zero_errors[col] = count


if dot_zero_errors:

    raise ValueError(
        "Unexpected .0 formatting in integer fields:\n"
        + str(dot_zero_errors)
    )


print(
    "PASS: No unnecessary .0 in integer fields."
)


# ============================================================
# 31. EXACT DECIMAL FIDELITY
# ============================================================

print("\n--- DECIMAL VALUE FIDELITY ---")


for col in DECIMAL_COLUMNS:

    expected_values = (
        expected_df[col]
        .to_numpy(
            dtype=float
        )
    )


    exported_values = (
        pd.to_numeric(
            raw_csv_df[col]
            .where(
                raw_csv_df[col] != "",
                np.nan
            ),
            errors="coerce"
        )
        .to_numpy(
            dtype=float
        )
    )


    if not np.allclose(
        expected_values,
        exported_values,
        rtol=0,
        atol=0,
        equal_nan=True
    ):

        raise ValueError(
            f"{col}: decimal fidelity failed."
        )


    print(
        f"PASS: {col} decimals preserved exactly."
    )


# ============================================================
# 32. SCIENTIFIC NOTATION CHECK
# ============================================================

with open(
    csv_file,
    "r",
    encoding="utf-8"
) as f:

    csv_text = f.read()


scientific_count = (
    csv_text.lower().count("e-")
    +
    csv_text.lower().count("e+")
)


print(
    "\nScientific notation occurrences:",
    scientific_count
)


if scientific_count != 0:

    raise ValueError(
        "Unexpected scientific notation found."
    )


print(
    "PASS: No scientific notation."
)


# ============================================================
# 33. FINAL CSV SEQN CHECK
# ============================================================

final_seqn = pd.to_numeric(
    raw_csv_df["SEQN"],
    errors="raise"
)


if not (
    final_seqn.min() == 62161
    and
    final_seqn.max() == 71916
    and
    int(
        final_seqn.duplicated()
        .sum()
    ) == 0
):

    raise ValueError(
        "Final CSV SEQN validation failed."
    )


print(
    "PASS: Final CSV SEQN is valid."
)


# ============================================================
# 34. FINAL SUMMARY
# ============================================================

print(
    "\n"
    + "=" * 75
)


print(
    "DEMO_G CONVERSION COMPLETE"
)


print(
    "=" * 75
)


print(
    "Rows:",
    f"{len(df):,}"
)


print(
    "Columns:",
    len(df.columns)
)


print(
    "Original tiny-value artifacts:",
    f"{tiny_total:,}"
)


print(
    "Total validated zero restorations:",
    f"{correction_total:,}"
)


print(
    "Genuine decimal variables:",
    DECIMAL_COLUMNS
)


print(
    "Unexpected XPT -> CSV differences: 0 expected"
)


print(
    "Missing-position differences: 0 expected"
)


print(
    "Tiny values remaining: 0 expected"
)


print(
    "Unnecessary .0 in integer fields: 0 expected"
)


print(
    "Scientific notation: 0 expected"
)

NHANES 2011-2012 DEMO_G XPT -> CSV
Rows: 9,756
Columns: 48
PASS: Structure = 9,756 rows x 48 columns.

Expected genuine decimal variables:
['WTINT2YR', 'WTMEC2YR', 'INDFMPIR']

--- ORIGINAL XPT TINY-ZERO CHECK ---
Tiny artifact occurrences: 19,277

Tiny values by variable:
DMDHHSZE    7060
DMDHHSZA    6184
DMDHHSZB    4819
WTMEC2YR     418
RIDAGEYR     392
DMDEDUC3     282
INDFMPIR      80
RIDAGEMN      34
RIDEXAGM       8

PASS: Exact 19,277-value tiny-zero pattern validated.

--- ZERO RESTORATION ---
RIDAGEYR: 392 restored to 0
RIDAGEMN: 34 restored to 0
RIDEXAGM: 8 restored to 0
DMDEDUC3: 282 restored to 0
WTMEC2YR: 418 restored to 0
INDFMPIR: 80 restored to 0
DMDHHSZA: 6,184 restored to 0
DMDHHSZB: 4,819 restored to 0
DMDHHSZE: 7,060 restored to 0

PASS: 19,277 validated zero values restored.
PASS: No tiny artifacts remain after restoration.

--- FRACTIONAL VALUE CHECK ---
WTINT2YR: 9,756 fractional observations
WTMEC2YR: 9,338 fractional observations
INDFMPIR: 7,513 fractional obs